In [ ]:
try:
    import pygeohash as pgh
except ImportError:
    !pip install pygeohash
    import pygeohash as pgh

import sys
import os
sys.path.append('../../src')  # repo-relative: notebooks/pipeline/../../src

import pandas as pd
import numpy as np
from config import *

train_df = pd.read_csv(os.path.join(PROC_DIR, 'train_step03.csv'))
test_df = pd.read_csv(os.path.join(PROC_DIR, 'test_step03.csv'))

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)

In [ ]:
# Get unique geohashes from both train and test
unique_geohashes = set(train_df['geohash'].unique()).union(set(test_df['geohash'].unique()))

# Decode geohash to latitude and longitude
geohash_coords = {}
for gh in unique_geohashes:
    lat, lon = pgh.decode(gh)
    geohash_coords[gh] = (float(lat), float(lon))

# Map to train and test
train_df['latitude'] = train_df['geohash'].map(lambda x: geohash_coords[x][0])
train_df['longitude'] = train_df['geohash'].map(lambda x: geohash_coords[x][1])

test_df['latitude'] = test_df['geohash'].map(lambda x: geohash_coords[x][0])
test_df['longitude'] = test_df['geohash'].map(lambda x: geohash_coords[x][1])

print(train_df[['geohash', 'latitude', 'longitude']].head())

print("\nTrain Latitude - Min:", train_df['latitude'].min(), "Max:", train_df['latitude'].max(), "Mean:", train_df['latitude'].mean())
print("Train Longitude - Min:", train_df['longitude'].min(), "Max:", train_df['longitude'].max(), "Mean:", train_df['longitude'].mean())

In [ ]:
# Extract geohash prefix
train_df['geohash_prefix'] = train_df['geohash'].str[:GEOHASH_PREFIX_LEN]
test_df['geohash_prefix'] = test_df['geohash'].str[:GEOHASH_PREFIX_LEN]

unique_full = train_df['geohash'].nunique()
unique_prefix = train_df['geohash_prefix'].nunique()

print(f"Unique full geohash cells (Train): {unique_full}")
print(f"Unique geohash prefix cells (Train): {unique_prefix}")

print("\nTop 10 most common prefixes in Train:")
print(train_df['geohash_prefix'].value_counts().head(10))

In [ ]:
# Leave-one-out target encoding for geohash in train
# Calculate total demand sum and count per geohash
geohash_stats = train_df.groupby('geohash')['demand'].agg(['sum', 'count'])

# Join stats to train
train_df = train_df.join(geohash_stats, on='geohash')

# Calculate Leave-One-Out mean
global_mean = train_df['demand'].mean()

# LOO formula: (sum of demand for this geohash - current row demand) / (count of rows for this geohash - 1)
# For geohash cells with only one row, use the global mean demand as the fallback
train_df['geohash_target_enc'] = np.where(
    train_df['count'] > 1,
    (train_df['sum'] - train_df['demand']) / (train_df['count'] - 1),
    global_mean
)

# Clean up temporary columns
train_df = train_df.drop(columns=['sum', 'count'])

print(train_df[['geohash', 'demand', 'geohash_target_enc']].head())
print("\nMean of geohash_target_enc:", train_df['geohash_target_enc'].mean())
print("Std of geohash_target_enc:", train_df['geohash_target_enc'].std())

In [ ]:
# Leave-one-out mean demand per geohash_prefix on train
# (matches the LOO discipline used for geohash_target_enc two cells above —
#  a plain groupby().mean() applied back onto the same rows that produced it
#  would let each row see its own demand baked into its own encoding)
prefix_stats = train_df.groupby('geohash_prefix')['demand'].agg(prefix_sum='sum', prefix_count='count')
train_df = train_df.join(prefix_stats, on='geohash_prefix')

train_df['geohash_prefix_enc'] = np.where(
    train_df['prefix_count'] > 1,
    (train_df['prefix_sum'] - train_df['demand']) / (train_df['prefix_count'] - 1),
    global_mean
)
train_df = train_df.drop(columns=['prefix_sum', 'prefix_count'])

# Test uses the plain (non-LOO) train-derived prefix mean, with fallback to global mean for unseen prefixes
prefix_mean_demand = train_df.groupby('geohash_prefix')['demand'].mean().to_dict()
test_df['geohash_prefix_enc'] = test_df['geohash_prefix'].map(prefix_mean_demand).fillna(global_mean)

print(train_df[['geohash_prefix', 'geohash_prefix_enc']].head())
print(f"\nNulls in Train geohash_prefix_enc: {train_df['geohash_prefix_enc'].isnull().sum()}")

In [ ]:
# Apply geohash mean encoding to test
# Simple mean per geohash on train
geohash_mean_train = train_df.groupby('geohash')['demand'].mean()

# Vectorised fallback logic
direct_enc = test_df['geohash'].map(geohash_mean_train)
fallback_enc = test_df['geohash_prefix'].map(prefix_mean_demand)

mask_direct = direct_enc.notnull()
mask_fallback = (~mask_direct) & fallback_enc.notnull()
mask_global = (~mask_direct) & (~mask_fallback)

geohash_used = mask_direct.sum()
prefix_used = mask_fallback.sum()
global_used = mask_global.sum()

test_df['geohash_target_enc'] = direct_enc.fillna(fallback_enc).fillna(global_mean)

print(f"Test rows using direct geohash mean: {geohash_used}")
print(f"Test rows using prefix mean fallback: {prefix_used}")
print(f"Test rows using global mean fallback: {global_used}")
print(f"Total test rows: {len(test_df)}")
print(f"\nNulls in Test geohash_target_enc: {test_df['geohash_target_enc'].isnull().sum()}")

In [ ]:
import matplotlib.pyplot as plt

# Sample 10000 rows randomly
np.random.seed(42)
sample_df = train_df.sample(n=min(10000, len(train_df)))

plt.figure(figsize=(10, 5))
plt.scatter(sample_df['geohash_target_enc'], sample_df['demand'], alpha=0.1)

# Add reference line
max_val = max(sample_df['geohash_target_enc'].max(), sample_df['demand'].max())
min_val = min(sample_df['geohash_target_enc'].min(), sample_df['demand'].min())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect Encoding')

plt.title('Geohash Target Encoding vs Actual Demand')
plt.xlabel('Geohash Target Encoding')
plt.ylabel('Actual Demand')
plt.legend()

# Save plot
plot_path = os.path.join(BIVARIATE_PLOTS_DIR, 'geohash_encoding_quality.png')
plt.savefig(plot_path, bbox_inches='tight')
plt.show()

In [ ]:
# Drop raw geohash and prefix columns
train_df = train_df.drop(columns=['geohash', 'geohash_prefix'])
test_df = test_df.drop(columns=['geohash', 'geohash_prefix'])

print("Columns after dropping:")
print("Train:", train_df.columns.tolist())
print("Test:", test_df.columns.tolist())

In [ ]:
# Save
train_path = os.path.join(PROC_DIR, 'train_step04.csv')
test_path = os.path.join(PROC_DIR, 'test_step04.csv')

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Train saved to {train_path} with shape {train_df.shape}")
print(f"Test saved to {test_path} with shape {test_df.shape}")